In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVR
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
import shap

In [ ]:
df = pd.read_csv("output/cleaned_merged.csv")
df = df.drop(columns=['brightness_subjective','contrast_subjective','colorfulness_subjective','sharpness_subjective','SD','image_name'])
print(df.columns)
total = len(df)
koniq_entries = (df['dataset']==('koniq')).sum()
biq_entries = (df['dataset']==('biq2021')).sum()
print(f"total rows: {total}")
print(f"koniq : biq ratio - {(koniq_entries / biq_entries):.2f}")

In [ ]:
X = df.drop(columns=['MOS'])
y = df['MOS']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42, stratify = df['dataset'])

In [ ]:
X_train, X_test, y_train, y_test = X_train.drop(columns='dataset'), X_test.drop(columns='dataset'), y_train.drop(columns='dataset'), y_test.drop(columns='dataset')

In [ ]:
model = SVR()
print(np.logspace(-2,3,6))
param_grid = [{'kernel': ['rbf'],
              'C': np.logspace(-2,3,6),
             'epsilon': [0.001,0.01,0.1,0.2,0.5,1.0],
             'gamma': ["scale", "auto", 0.1, 0.01, 0.001, 0.0001]
             },
             # {'kernel': ['poly'],
             #  'C': np.logspace(-2,3,6),
             # 'epsilon': [0.001,0.01,0.1,0.2,0.5,1.0],
             # },
             # {'kernel': ['sigmoid'],
             #  'C': np.logspace(-2,3,6),
             # 'epsilon': [0.001,0.01,0.1,0.2,0.5,1.0],
             # }
             ]

grid_search = GridSearchCV(
    estimator = model,
    param_grid=param_grid,
    cv=5,
    verbose=3,
    n_jobs = -2
)

In [ ]:
# grid_search.fit(X_train, y_train)
# results_df = pd.DataFrame(grid_search.cv_results_)
# results_df.to_csv("grid_search_results.csv", index=False)

In [ ]:
model = model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)

print(f"R² Score: {r2:.4f}")
print(f"MAE:      {mae:.4f}")
print(f"RMSE:     {rmse:.4f}")

# Shapley Values

In [ ]:
explainer = shap.KernelExplainer(model.predict, shap.sample(X_train, 10), seed=42)
shap_values = explainer(shap.sample(X_test, 10))



In [ ]:
shap.plots.beeswarm(shap_values, max_display=30)
